contextlib 是 Python 标准库中用于简化上下文管理器（Context Manager）创建的模块

在深度学习和分布式训练（如 PyTorch）中，它至关重要，主要用于管理资源（如 GPU 显存、锁、文件句柄）或临时修改状态（如禁用梯度计算、切换随机种子），并确保在代码块执行完毕或发生异常时，自动恢复现场。

## 核心概念：什么是上下文管理器？
- 上下文管理器是通过 with 语句使用的对象，它定义了进入代码块时做什么（__enter__）和退出时做什么（__exit__）。

### 传统写法

In [ ]:
class MyResource:
    def __enter__(self):
        print("获取资源")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("释放资源")
        if exc_type:
            print(f"发生错误: {exc_val}")
        return False # 不吞掉异常

# 使用
with MyResource():
    print("做事...")

#### contextlib 的作用：让你不用写类，直接用函数或装饰器实现同样的功能。

## contextlib.contextmanager (最常用)

在编写分布式训练代码或复杂的深度学习管道时，凡是涉及“临时改变状态”或“需要确保清理资源”的地方，首选 contextlib。它是写出健壮、不易出错代码的必备工具。

将生成器函数转换为上下文管理器。
- yield 之前的代码 = __enter__ (进入时执行)
- yield 本身 = 返回值传递给 as 变量
- yield 之后的代码 = __exit__ (退出时执行，自动处理异常)

In [ ]:
from contextlib import contextmanager
import torch

@contextmanager
def temporary_cuda_device(device_id):
    """临时切换当前 CUDA 设备，结束后自动切回"""
    original_device = torch.cuda.current_device()
    print(f"切换设备到: {device_id}")
    torch.cuda.set_device(device_id)
    
    try:
        yield  # 这里暂停，执行 with 块内的代码
    finally:
        # 无论 with 块内是否报错，这里都会执行
        print(f"恢复设备到: {original_device}")
        torch.cuda.set_device(original_device)

# 使用
with temporary_cuda_device(1):
    print(f"当前在设备: {torch.cuda.current_device()}")
    # 做一些操作...
    # 如果这里报错，finally 依然会执行恢复逻辑

print(f"出来后设备: {torch.cuda.current_device()}")

## ExitStack (动态管理多个资源)

当你需要在运行时动态决定打开多少个文件、锁或连接时，嵌套多个 with 会很丑（"箭头型代码"）。ExitStack 可以编程式地管理它们

In [ ]:
from contextlib import ExitStack

def load_multiple_models(file_paths):
    with ExitStack() as stack:
        files = []
        for path in file_paths:
            # 动态打开文件，并注册关闭操作
            f = stack.enter_context(open(path, 'r'))
            files.append(f)
        
        # 在这里处理所有文件
        # ...
        
        # 当离开 with 块时，stack 会自动按相反顺序关闭所有文件
        # 即使中间某个文件处理报错，已打开的文件也会被关闭

# 避免了这种丑陋的嵌套：
# with open(p1) as f1:
#     with open(p2) as f2:
#         with open(p3) as f3:
#             ...

## PyTorch 内部大量使用了 contextlib 来实现关键功能。

### torch.no_grad() (禁用梯度)

这是最常见的用法。用于推理阶段或冻结部分网络，节省显存和计算。
- 原理：进入时设置全局标志位 grad_mode=False，退出时恢复。

In [ ]:
import torch

model.eval()
with torch.no_grad():
    output = model(input_data) 
    # 这里不会构建计算图，output.requires_grad 为 False
# 退出后，梯度追踪自动恢复（如果之前是开启的）

### torch.autocast() (混合精度训练) 
- 自动根据操作选择 FP16 或 FP32，加速训练并节省显存。

In [ ]:
with torch.autocast(device_type="cuda", dtype=torch.float16):
    output = model(input_data)
    loss = criterion(output, target)
# 退出后自动恢复默认精度

### 自定义训练循环中的日志/计时
- 你可以用 @contextmanager 封装一个计时器，自动记录每个 Epoch 或 Layer 的耗时。

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(name):
    start = time.time()
    yield
    end = time.time()
    print(f"[{name}] 耗时: {end - start:.4f} 秒")

# 使用
for epoch in range(10):
    with timer(f"Epoch {epoch}"):
        # 训练代码
        time.sleep(0.5) 

## 为什么它很重要？

1. 安全性 (Safety)：
- 确保资源（显存、文件、锁）一定会被释放，即使在 with 块内部发生了崩溃或异常 (Exception)。
- 防止显存泄漏（OOM）或死锁。
2. 可读性 (Readability)：
- 资源的“获取”和“释放”逻辑在视觉上被包裹在一起，代码意图清晰。
- 减少了大量的 try...finally 样板代码。
3. 状态隔离 (Isolation)：
- 临时修改全局状态（如随机数种子 torch.manual_seed、设备 cuda.set_device、精度模式），用完自动还原，避免对后续代码产生副作用。